# Load Packages

The following cell loads the necessary packages to run ezTrackRT. It does not need to be changed by the user.

In [ ]:
%load_ext autoreload
%autoreload 2

from matplotlib import pyplot as plt
import RT_functions as rt

# Create Video Instance

The following cell connects ezTrackRT to the video camera and starts the video stream. The default camera (`src=0`) is used. This cell does not need to be changed unless a different camera is being used.

In [ ]:
vid = rt.Video(src=0)
vid.start()

# Define Cropping Bounds

The following cell allows you to define the portion of the video frame that will be used for location tracking and freezing analysis. Use the box selection tool (square with a + sign) to draw a rectangle around the area you want to keep. Double-click to define one corner of the area to be cropped, and double-click again to define the opposite corner. The selected region will be used to crop each incoming frame before it is processed for location tracking or freezing analysis.

In [ ]:
%output size = 100

vid.crop_define()

# Define Distance Scale

The following cell allows you to define a physical distance scale that can be used to report distances during location tracking or freezing analysis. Enter a name for the desired distance unit (e.g., `'cm'`) and the known distance between two points in that unit. Use the point selection tool (the icon with three circles) to mark the two corresponding points on the video frame. Click once to place the first point, and click again to place the second point. Here, `name = 'cm'` and `dist = 10` are examples indicating that the distance between the two selected points is 10 cm; these values should be changed to match the physical reference being used. This step is only needed if you would like distances to be reported in a physical unit rather than pixels.

In [ ]:
%output size = 100

vid.distance_define(name = 'cm', dist=10)

# Select Regions to Exclude from Tracking

The following cell allows you to define one or more regions of the video frame to exclude from location tracking and freezing analysis. Use the polygon drawing tool (the polygon-shaped icon) to outline the region(s) you want to exclude. Double-click to start and finish a region, and single-click to add additional vertices to the region. Excluded regions will not be considered when ezTrackRT determines the animal's location or freezing behavior. This step is optional and is useful for excluding stationary objects or other areas of the frame that could interfere with location tracking or freezing analysis.

In [ ]:
%output size = 100

vid.mask_define()

# Select Regions of Interest

The following cell allows you to define one or more regions of interest (ROIs) within the video frame. Enter the desired ROI name(s) in names (e.g., `names=['left']`). For multiple ROIs, separate them using commas and ensure each name is in quotation marks. Use the polygon drawing tool (the polygon-shaped icon) to outline each ROI. Double-click to start and finish a region, and single-click to add additional vertices. ROIs can be used to generate ROI-specific data (e.g., how much time the animal spends freezing in ROI 1 vs. ROI 2).

In [ ]:
%output size = 100

vid.roi_define(names=['left'])

# Save cropping/roi/masking parameters

The following cell saves the cropping, ROI, masking, and distance-scale settings. The saved file is a .pickle file and should be given a descriptive filename. Change `filename` to the file directory ending in the desired filename where you want to save the metadata/parameters file. Include the .pickle extension in the filename.

**Windows:** Use a path such as:
`filename = r'C:\Users\YourName\Documents\ezTrackRT\parameters.pickle'`

**macOS:** Use a path such as:
`filename = '/Users/YourName/Documents/ezTrackRT/parameters.pickle'`

**Linux:** Use a path such as:
`filename = '/home/YourName/Documents/ezTrackRT/parameters.pickle'`

Replace the example path with the folder and filename you want to use. All folders within the path must already exist; `params_save()` does not create new folders. Once the desired path has been entered, run the following cell to save the parameters.

In [ ]:
filename = '/Users/mcter/OneDrive/Documents/test'

vid.params_save(file=filename)

# Load cropping/roi/masking parameters

The following cell loads previously saved cropping, ROI, masking, and distance-scale parameters. This allows you to reuse the same processing setup without redefining these parameters manually. Change `filename` to the location of the .pickle parameter file you previously saved.

**Windows:** Use a path such as:
`filename = r'C:\Users\YourName\Documents\ezTrackRT\parameters.pickle'`

**macOS:** Use a path such as:
`filename = '/Users/YourName/Documents/ezTrackRT/parameters.pickle'`

**Linux:** Use a path such as:
`filename = '/home/YourName/Documents/ezTrackRT/parameters.pickle'`

Replace the example path with the location and filename of your saved parameter file. The file must already exist at the specified location. Once the correct path has been entered, run the following cell to load the parameters.

In [ ]:
filename = '/Users/mcter/OneDrive/Documents/test'

vid.params_load(file=filename)

# Set Reference Frame for Tracking

The following cell creates a reference frame that can be used for location tracking and freezing analysis. While the video is running, `vid.ref_create()` collects frames for the specified duration and calculates their average to create a reference frame representing the background of the recording area. Here, `secs=1` means that frames collected over 1 second are used to create the reference frame. Adjust this value as needed. `print_sts=False` prevents the progress of reference-frame creation from being printed. The second line displays the resulting reference frame. Check that the reference frame accurately represents the background of the recording area and does not contain the animal or other objects that should be excluded from the background.

In [ ]:
vid.ref_create(secs=1, print_sts=False)

plt.imshow(vid.ref, cmap='gray')

# Set Desired Analysis Type

Select whether location tracking or freezing analysis is desired, and if the latter, which method will be used to calculate freezing. Set `vid.freeze_method` to `'distance'` to calculate freezing based on the animal's center-of-mass movement (similar to the location tracking algorithm), `'pixel'` to calculate freezing based on pixel fluctuations between consecutive frames, or `None` to disable freezing analysis and instead perform location tracking. If `None` is selected, only location tracking parameters will be used below.

In [ ]:
vid.freeze_method = None

# Set Analysis-Specific Parameters

**The following cell sets parameters used for location tracking and distance-based freezing analysis.** These settings determine how the animal is identified and localized within each video frame and should be changed as desired.

* `track_method` determines how differences between the current frame and the reference frame are calculated. Set to `'abs'` to detect changes regardless of whether the animal is lighter or darker than the background. Alternatively, use `'light'` if the animal is lighter than the background or `'dark'` if the animal is darker than the background.
* `track_thresh` determines the percentile threshold used to remove smaller pixel differences before calculating the animal's location. For example, `95` removes difference values below the 95th percentile.
* `track_window_use` determines whether a window around the animal's previous location is used to give greater weight to that area when determining its current location.
* `track_window_sz` sets the width and height, in pixels, of the tracking window when `track_window_use=True`.
* `track_window_wt` determines how strongly the tracking window is weighted, from 0 to 1. This parameter only affects tracking when `track_window_use=True`.
* `track_rmvwire` determines whether morphological processing is used to reduce the effect of wires or similar thin structures on location tracking.
* `track_rmvwire_krn` sets the size of the morphological kernel used for wire removal. The kernel should be larger than the wire but smaller than the animal.

In [ ]:
if vid.freeze_method is None or vid.freeze_method == 'distance':
    vid.track_method = 'abs'
    vid.track_thresh = 95
    vid.track_window_use = False
    vid.track_window_sz = 100
    vid.track_window_wt = 0.9
    vid.track_rmvwire = False
    vid.track_rmvwire_krn = 10

**The following cell sets parameters used for freezing analysis (distance- and pixel-based).** These settings determine how the animal's freezing behaviour is identified and quantified within each video frame and should be changed as desired.

* `freeze_buffer_size` determines the number of consecutive frames for which freezing-related measurements are stored. More specifically, the values represent center-of-mass movement when using the distance method or pixel fluctuations when using the pixel method. These measurements are used to determine whether the animal is currently freezing.
* `freeze_thresh` sets the threshold used to determine whether the animal is freezing based on the values stored in the buffer.
* `freeze_thresh_method` determines how the values stored in the buffer are evaluated against `freeze_thresh`. Options are `'max'` or `'mean'`.
* `freeze_motion_thresh` sets the minimum pixel fluctuation considered to represent animal movement when using the pixel freezing method. Pixel fluctuations below this value are treated as background noise. **This parameter only applies to the pixel-based freezing method.**

In [ ]:
if vid.freeze_method is not None:
    vid.freeze_buffer_size = 30
    vid.freeze_thresh = 15
    vid.freeze_thresh_method = 'max'
    if vid.freeze_method == 'pixel':
        vid.freeze_motion_thresh = 25

# Initialize Analysis

This cell initializes the analysis specified by `freeze_method`. If `freeze_method` is set to `None`, location tracking is initiated. If `freeze_method` is set to `'distance'` or `'pixel'`, freezing analysis is initiated. In either case, `vid.start` must be run before analysis is begun.

In [ ]:
if vid.freeze_method is None:
    vid.track = True
if vid.freeze_method is not None:
    vid.freeze = True

# Initialize Data Writer
The following cell initializes the data writer, which prepares ezTrackRT to save video and analysis data during the experiment. `dpath` specifies the folder where the output files will be saved. The writer is initialized but does not begin saving until `vid.writer_start()` is called. Change `dpath` to the desired output folder on your computer. This should be done AFTER any cropping is performed, as a video instance of the current frame size will be initiated.

**Windows:** Use a path such as:
dpath = r'C:\Users\YourName\Documents\ezTrackRT_data'

**macOS:** Use a path such as:
dpath = '/Users/YourName/Documents/ezTrackRT_data'

**Linux:** Use a path such as:
dpath = '/home/YourName/Documents/ezTrackRT_data'

In [ ]:
vid.writer_init(dpath="C:/Users/mcter/OneDrive/Documents/")

# Start Data Writer

The following cell signals the data writer to begin saving video and analysis data. The data writer must first be initialized using `vid.writer_init()`. Once started, data will be saved to the output directory specified by `dpath` in `vid.writer_init()`.

In [ ]:
vid.writer_start()

# Start Video Display.  Press 'q' to quit. 

The following cell starts the video display and begins processing frames using the tracking/freezing settings defined above. Press q to stop the video display and end processing. `show_xy=True` displays the animal's tracked center-of-mass position (for location tracking and distance-based freezing only). `show_dif=True` displays the difference image used for the location/freezing analysis. For location tracking and distance-based freezing analysis, this is the pixel-wise difference between the current frame and the reference frame. For pixel-based freezing analysis, this is the pixel fluctuations between the current and previous frame. The final line, `vid.track_dist`, displays the most recently calculated distance traveled between consecutive frames (for location tracking and distance-based freezing only).

In [ ]:
# this cell originally had a typo

vid.display(show_xy=True, show_dif=True)

In [ ]:
vid.track_dist

# Stop Saving Video and Release Camera
The following cell stops video processing, stops the data writer, and releases the camera. Run this cell when the experiment is finished. This should be the final cell run before closing the notebook or reconnecting to a different camera. See `vid.stop` and `vid.writer_stop` for more nuanced control.

In [ ]:
vid.release()